In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

In [2]:
CONFIG = {
    "batch_size": 64,
    "epochs": 50,
    "lr": 0.003,
    "weight_decay": 0.0001,
    "label_smoothing": 0.1,
    "num_workers": 2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 23,
}

In [3]:
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1),
        shear=5
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=train_transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"])
test_loader  = DataLoader(test_dataset,  batch_size=256,                  shuffle=False, num_workers=CONFIG["num_workers"])

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 508kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.58MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.38MB/s]


In [4]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            # Block 1: 1 -> 32 channels, 28x28 -> 14x14
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            # Block 2: 32 -> 64 channels, 14x14 -> 7x7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            # Block 3: 64 -> 128 channels, 7x7 -> 3x3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            # Block 3: 128 -> 256 channels, 3x3 -> 1x1
            nn.Conv2d(128, 256, kernel_size=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),                        # 256 * 1 * 1 = 256
            nn.Linear(256 * 1 * 1, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [5]:
model = Model().to(CONFIG["device"])
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

Model parameters: 160,842


In [6]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

# Warmup for 5 epochs, then cosine decay
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=CONFIG["lr"],
    steps_per_epoch=len(train_loader),
    epochs=CONFIG["epochs"],
    pct_start=0.1,          # 10% warmup
    anneal_strategy="cos",
)

criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

In [7]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, device, tta=False):
    """Evaluate with optional Test-Time Augmentation."""
    model.eval()
    correct, total = 0, 0

    tta_transforms = [
        transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]),
        transforms.Compose([transforms.RandomRotation(5),  transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]),
        transforms.Compose([transforms.RandomAffine(0, translate=(0.05, 0.05)), transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]),
    ]

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)

    return correct / total

In [8]:
best_acc = 0.0
history = {"train_loss": [], "train_acc": [], "test_acc": []}

print("\n" + "="*60)
print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Train Acc':>10}  {'Test Acc':>10}  {'LR':>10}")
print("="*60)

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, CONFIG["device"]
    )
    test_acc = evaluate(model, test_loader, CONFIG["device"])

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    current_lr = scheduler.get_last_lr()[0]

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "mnist_best.pth")
        marker = " ✓ BEST"
    else:
        marker = ""

    print(f"{epoch:>6}  {train_loss:>10.4f}  {train_acc*100:>9.2f}%  {test_acc*100:>9.2f}%  {current_lr:>10.6f}{marker}")

print("="*60)
print(f"\nBest test accuracy: {best_acc*100:.2f}%")


 Epoch  Train Loss   Train Acc    Test Acc          LR
     1      1.5776      52.87%      95.11%    0.000395 ✓ BEST
     2      0.8644      87.52%      97.11%    0.001115 ✓ BEST
     3      0.7545      92.05%      98.13%    0.002006 ✓ BEST
     4      0.7104      93.52%      98.53%    0.002725 ✓ BEST
     5      0.6858      94.29%      98.75%    0.003000 ✓ BEST
     6      0.6660      95.03%      98.78%    0.002996 ✓ BEST
     7      0.6530      95.43%      98.84%    0.002985 ✓ BEST
     8      0.6437      95.54%      98.91%    0.002967 ✓ BEST
     9      0.6410      95.56%      99.14%    0.002942 ✓ BEST
    10      0.6323      95.84%      99.07%    0.002910
    11      0.6307      95.84%      99.10%    0.002870
    12      0.6261      95.96%      98.97%    0.002824
    13      0.6232      96.08%      99.05%    0.002772
    14      0.6203      96.12%      99.06%    0.002713
    15      0.6145      96.34%      99.02%    0.002649
    16      0.6124      96.50%      99.22%    0.002579 ✓

In [9]:
print("\nLoading best model for final evaluation...")
model.load_state_dict(torch.load("mnist_best.pth", map_location=CONFIG["device"]))
final_acc = evaluate(model, test_loader, CONFIG["device"])
print(f"Final test accuracy: {final_acc*100:.2f}%")


Loading best model for final evaluation...
Final test accuracy: 99.43%


In [10]:
def confusion_matrix(model, loader, device, num_classes=10):
    model.eval()
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(1).cpu().numpy()
            for true, pred in zip(labels.numpy(), preds):
                matrix[true][pred] += 1
    return matrix

cm = confusion_matrix(model, test_loader, CONFIG["device"])
print("\nConfusion Matrix (rows=true, cols=predicted):")
print("   " + " ".join(f"{i:4}" for i in range(10)))
for i, row in enumerate(cm):
    errors = sum(row) - row[i]
    print(f"{i}: " + " ".join(f"{v:4}" for v in row) + f"  [{errors} errors]")

per_class_acc = cm.diagonal() / cm.sum(axis=1)
print("\nPer-class accuracy:")
for i, acc in enumerate(per_class_acc):
    print(f"  Digit {i}: {acc*100:.1f}%")


Confusion Matrix (rows=true, cols=predicted):
      0    1    2    3    4    5    6    7    8    9
0:  980    0    0    0    0    0    0    0    0    0  [0 errors]
1:    0 1132    0    1    0    1    0    1    0    0  [3 errors]
2:    1    0 1025    2    0    0    1    3    0    0  [7 errors]
3:    0    0    0 1008    0    1    0    0    1    0  [2 errors]
4:    0    0    0    0  976    0    2    0    0    4  [6 errors]
5:    1    0    0    3    0  885    2    1    0    0  [7 errors]
6:    2    1    0    0    2    3  949    0    1    0  [9 errors]
7:    0    4    2    0    0    1    0 1020    0    1  [8 errors]
8:    0    0    2    1    0    1    0    0  968    2  [6 errors]
9:    0    0    0    0    4    1    0    3    1 1000  [9 errors]

Per-class accuracy:
  Digit 0: 100.0%
  Digit 1: 99.7%
  Digit 2: 99.3%
  Digit 3: 99.8%
  Digit 4: 99.4%
  Digit 5: 99.2%
  Digit 6: 99.1%
  Digit 7: 99.2%
  Digit 8: 99.4%
  Digit 9: 99.1%
